# Feature generation | Google Earth Engine

Public adaptation of `src/02_station_features/01_CAL_make_gg_ee_rswq_clean_2562_2568_b100_v4.ipynb` in the team archive. Use this to inspect how features were produced or to regenerate them with your own Earth Engine project and authorized inputs. To use the existing exports, go directly to the prediction notebooks instead. Exports are disabled by default. Details: [Feature sources and Drive inputs](https://github.com/Boonyaratt/sentinel2-water-quality/blob/main/docs/drive-features-th.md).


# Sentinel-2 Feature Extraction for EPO14 Water Quality

This clean notebook extracts Sentinel-2 tabular features for EPO14 water-quality observations. It supports BE 2562-2568, exports Earth Engine results in chunks, merges the chunks, and builds QA/model-ready CSV files.

Scientific note: Sentinel-2 provides optical proxy information. Turbidity and SS are more directly linked to optical signals, while DO, BOD, NH3, TCB, FCB, and WQI should be treated as indirect proxy predictions.


## Verified Colab run without export · 2026-11-13

The notebook processed all 826 field observations with water mask v4. A Colab test retrieved live Earth Engine results in small requests and ran the QA and candidate-table cells with batch export and Drive copy disabled.

| Station table before the Context join | Archived ZIP v2 rows | Live Colab rows |
| --- | ---: | ---: |
| Full (all cleaned field observations) | 826 | 826 |
| Strict candidates | 175 | 174 |
| Relaxed candidates | 340 | 342 |

Across 44 feature and QA columns, 811 rows matched the archive and 15 differed. Six of the 15 selected a different scene ID. The live run still reported more water than clear pixels in 83 rows.

The historical model uses the archived relaxed table **after** joining environmental context (340 rows × 226 columns). The [station dataset guide and run log](https://github.com/Boonyaratt/sentinel2-water-quality/blob/main/docs/station-features-run-log.md) lists the filters, comparison method, and water-mask checks.


## Session 0. Environment Setup

Install and import packages used for Earth Engine extraction, mapping, and QA plots.


In [ ]:
%pip install -q --no-cache-dir "geemap==0.36.6" folium seaborn xyzservices python-box


In [ ]:
from IPython.display import display
from pathlib import Path
import re
import shutil

import ee
import folium
import geemap.foliumap as geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from google.colab import drive, files

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)


## Session 1. Earth Engine Authentication

Authenticate and initialize Google Earth Engine. Change `EE_PROJECT` if needed.


In [ ]:
EE_PROJECT = "YOUR_EARTH_ENGINE_PROJECT_ID"
if EE_PROJECT == "YOUR_EARTH_ENGINE_PROJECT_ID":
    raise ValueError("Set EE_PROJECT to a project you can use in Earth Engine.")

try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized:", EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine authenticated and initialized:", EE_PROJECT)


## Session 2. Project Configuration

Set input/output paths, year range, Sentinel-2 extraction settings, and Drive export folders.


In [ ]:
BASE = Path("/content")

# This notebook is designed for BE 2562-2568 (AD 2019-2025).
YEAR_BE_MIN = 2562
YEAR_BE_MAX = 2568
INPUT_FILENAME = f"wq_14_{YEAR_BE_MIN}_{YEAR_BE_MAX}_rs_ready.csv"
IN_CSV = BASE / INPUT_FILENAME

OUT_DIR = BASE / "tabular_data_14/03_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sentinel-2 extraction settings.
SCALE_M = 20
BUFFER_M = 100
PRIMARY_WINDOW_DAYS = 3
FALLBACK_WINDOW_DAYS = 7
PRIMARY_CLOUD_PCT = 80
FALLBACK_CLOUD_PCT = 90
WATER_NDWI_THRESHOLD = 0.0
WATER_MNDWI_THRESHOLD = 0.1
MIN_CANDIDATE_WATER_PIXELS = 3

# Earth Engine Drive export folders must be flat names. Do not use slashes here.
CHUNK_SIZE = 50
RUN_TAG = f"epo14_{YEAR_BE_MIN}_{YEAR_BE_MAX}_b{BUFFER_M}_watermask_v4_s2formula"
DRIVE_CHUNK_FOLDER = f"rswq_exports_s2_chunks_{RUN_TAG}_rebuild"
DRIVE_CHUNK_PREFIX = f"sentinel2_features_{RUN_TAG}_chunk"

# Desired final nested Drive location. Colab copies files here after merge.
COPY_OUTPUTS_TO_FINAL_DRIVE = False
FINAL_DRIVE_DIR = Path("/content/drive/MyDrive/wqr_feature_rebuild/station_sentinel2")

print("Input CSV:", IN_CSV)
print("Output directory:", OUT_DIR)
print("Run tag:", RUN_TAG)
print("GEE flat export folder:", DRIVE_CHUNK_FOLDER)
print("Final Drive folder:", FINAL_DRIVE_DIR)


## Session 3. Load Field Observation Data

Upload the `rs_ready` CSV if it is not already in `/content`, then filter to BE 2562-2568 and create stable sample IDs.


In [ ]:
if not IN_CSV.exists():
    print(f"{IN_CSV.name} was not found in /content. Upload the rs_ready CSV now.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No CSV file was uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    IN_CSV = BASE / uploaded_name

print("Using input CSV:", IN_CSV)


In [ ]:
field = pd.read_csv(IN_CSV, encoding="utf-8-sig")
field["survey_date"] = pd.to_datetime(field["survey_date"], errors="coerce")

if "year_be" not in field.columns:
    field["year_be"] = field["survey_date"].dt.year + 543

field["year_be"] = pd.to_numeric(field["year_be"], errors="coerce")
field = field[field["year_be"].between(YEAR_BE_MIN, YEAR_BE_MAX)].copy()
field["year_be"] = field["year_be"].astype(int)
field["year_ad"] = field["year_be"] - 543

if "rs_use_status" in field.columns:
    field = field[field["rs_use_status"].fillna("keep").eq("keep")].copy()

if "station_canonical" not in field.columns:
    field["station_canonical"] = field["station"].astype(str)

if "station_original" not in field.columns:
    field["station_original"] = field["station_canonical"].astype(str)

field["survey_no"] = (
    field["survey_no"]
    .astype(str)
    .str.replace(r"/+", "_", regex=True)
    .str.strip("_")
)

field["sample_id"] = (
    field["station_canonical"].astype(str)
    + "_"
    + field["survey_no"].astype(str).str.replace("/", "_", regex=False)
    + "_"
    + field["survey_date"].dt.strftime("%Y%m%d")
)

required_cols = ["sample_id", "station_canonical", "survey_no", "survey_date", "latitude", "longitude"]
missing_required = [col for col in required_cols if col not in field.columns]
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

bad_rows = field[field[required_cols].isna().any(axis=1)].copy()
if len(bad_rows):
    display(bad_rows[required_cols].head(20))
    raise ValueError(f"Found {len(bad_rows)} rows with missing required metadata.")

print("Filtered shape:", field.shape)
print("Date range:", field["survey_date"].min(), "to", field["survey_date"].max())
print("Rows by BE year:")
display(field.groupby("year_be").size().rename("rows"))
print("Unique stations:", field["station_canonical"].nunique())

field[["sample_id", "station_original", "station_canonical", "survey_date", "latitude", "longitude"]].head()


## Session 4. Metadata, Labels, and Match Points

Define metadata and target columns. Export a lightweight point table for inspection.


In [ ]:
metadata_cols = [
    "sample_id",
    "station_survey_id",
    "canonical_station_survey_id",
    "station_original",
    "station_canonical",
    "year_be",
    "year_ad",
    "survey_no",
    "survey_date",
    "latitude",
    "longitude",
    "water_body",
    "coordinate_waterbody",
    "station_location",
    "coordinate_assignment_status",
]

target_cols = [
    "turbidity_NTU_clean",
    "SS_mg_l_clean",
    "DO_mg_l_clean",
    "BOD_mg_l_clean",
    "NH3_N_mg_l_clean",
    "TCB_MPN_100ml_clean",
    "FCB_MPN_100ml_clean",
    "WQI_clean",

]

wqi_aux_cols = [
    "WQI_DO_score_pcd5",
    "WQI_BOD_score_pcd5",
    "WQI_TCB_score_pcd5",
    "WQI_FCB_score_pcd5",
    "WQI_NH3_score_pcd5",
    "WQI_reported",
    "WQI_score_input_complete",
    "WQI_mean_subindex_pcd5",
    "WQI_mean_level_pcd5",
    "WQI_worst_parameter_level_pcd5",
    "WQI_adjustment_pcd5",
    "WQI_recalc_pcd5",
    "WQI_source",
    "WQI_recalc_status",
]

field_context_cols = [
    "pH_clean",
    "conductivity_clean",
    "salinity_ppt_clean",
    "temp_air_clean",
    "temp_water_clean",
]

# Optional observed columns used only if present in the source CSV.
# This lets the formula-comparison plots include Chl-a, Secchi depth, or TSI
# if future field tables contain those ground observations.
optional_formula_ground_truth_cols = [
    "chlorophyll_a_clean",
    "chlorophyll_a_ug_l_clean",
    "chlorophyll_a_mg_m3_clean",
    "chla_clean",
    "chla_ug_l_clean",
    "Chl_a_clean",
    "Chl-a_clean",
    "secchi_depth_m_clean",
    "secchi_m_clean",
    "SDD_m_clean",
    "SDD_clean",
    "transparency_m_clean",
    "TSI_clean",
    "trophic_state_index_clean",
]

field_context_cols = field_context_cols + [
    col for col in optional_formula_ground_truth_cols
    if col in field.columns and col not in field_context_cols
]

field_value_cols = target_cols + wqi_aux_cols + field_context_cols

points = field[metadata_cols + field_value_cols].copy()
points.to_csv(OUT_DIR / "rs_match_points.csv", index=False, encoding="utf-8-sig")


# B. Water Masking and Feature Extraction

## Session 6. Sentinel-2 Feature Definitions

Define bands and indices used in the exported feature table.


In [ ]:
S2_BANDS = [
    "B1", "B2", "B3", "B4", "B5", "B6", "B7",
    "B8", "B8A", "B9", "B11", "B12",
]

S2_INDEX_BANDS = [
    "NDWI", "MNDWI", "NDTI", "NDCI",
    "NDVI", "AWEIsh",
    "Red_Green", "Red_Blue", "NIR_Red",
    "B5_B4", "B6_B4", "B7_B4", "B8_B4",
]

# Sentinel-2 formula-derived proxy features adapted from scripts.js.
# These are exported as *_median because extraction summarizes water-masked pixels.
S2_SCRIPT_FORMULA_BANDS = [
    "ph_formula_s2_script",
    "turbidity_formula_s2_script",
    "salinity_formula_s2_script",
    "do_formula_s2_script",
    "chla_formula_s2_ndci_script",
    "secchi_formula_s2_script",
    "tsi_formula_s2_script",
]

S2_FEATURE_BANDS = S2_BANDS + S2_INDEX_BANDS + S2_SCRIPT_FORMULA_BANDS
S2_FEATURE_NAMES = [f"{band}_median" for band in S2_FEATURE_BANDS]

print("Feature bands:", S2_FEATURE_BANDS)
print("Script formula bands:", S2_SCRIPT_FORMULA_BANDS)
print("Buffer m:", BUFFER_M, "Scale m:", SCALE_M)


## Session 7. Convert Field Rows to Earth Engine Points

Convert each field observation into an Earth Engine point feature with station/date metadata.


In [ ]:
EE_PROPERTY_COLS = [
    "sample_id",
    "station_survey_id",
    "canonical_station_survey_id",
    "station_original",
    "station_canonical",
    "survey_no",
    "year_be",
    "year_ad",
    "water_body",
    "coordinate_waterbody",
    "coordinate_assignment_status",
]

def clean_ee_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value)
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")
    return str(value)

def row_to_ee_feature(row):
    lon = float(row["longitude"])
    lat = float(row["latitude"])

    props = {
        col: clean_ee_value(row[col])
        for col in EE_PROPERTY_COLS
        if col in row.index
    }

    props["field_date"] = row["survey_date"].strftime("%Y-%m-%d")
    props["lat"] = lat
    props["lon"] = lon

    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        props
    )

def pandas_to_ee_points(data):
    features = [
        row_to_ee_feature(row)
        for _, row in data.iterrows()
    ]
    return ee.FeatureCollection(features)

ee_points = pandas_to_ee_points(field)

print("Rows in field:", len(field))
print("Rows in ee_points:", ee_points.size().getInfo())

## Session 8. Sentinel-2 Preprocessing Functions

Apply Sentinel-2 SCL masking, calculate indices and ratios, build water/clear masks, and rank candidate scenes.


In [ ]:
def safe_ratio(numerator, denominator, name):
    # Avoid extreme/invalid ratios where the denominator is zero or masked.
    return numerator.divide(denominator).updateMask(denominator.neq(0)).rename(name)


def prep_s2(img):
    scl = img.select("SCL")

    clear = (
        scl.neq(0)    # no data
        .And(scl.neq(1))    # saturated/defective
        .And(scl.neq(3))    # cloud shadow
        .And(scl.neq(8))    # cloud medium probability
        .And(scl.neq(9))    # cloud high probability
        .And(scl.neq(10))   # thin cirrus
        .And(scl.neq(11))   # snow/ice
    )

    sr = img.select(S2_BANDS).multiply(0.0001).rename(S2_BANDS)

    coastal = sr.select("B1")
    blue = sr.select("B2")
    green = sr.select("B3")
    red = sr.select("B4")
    red_edge1 = sr.select("B5")
    red_edge2 = sr.select("B6")
    red_edge3 = sr.select("B7")
    nir = sr.select("B8")
    water_vapor = sr.select("B9")
    swir1 = sr.select("B11")
    swir2 = sr.select("B12")

    # Normalized-difference indices.
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    ndwi = green.subtract(nir).divide(green.add(nir)).rename("NDWI")
    mndwi = green.subtract(swir1).divide(green.add(swir1)).rename("MNDWI")
    ndti = red.subtract(green).divide(red.add(green)).rename("NDTI")
    ndci = red_edge1.subtract(red).divide(red_edge1.add(red)).rename("NDCI")

    # Simple band ratios requested for the tabular feature set.
    red_green = safe_ratio(red, green, "Red_Green")
    red_blue = safe_ratio(red, blue, "Red_Blue")
    nir_red = safe_ratio(nir, red, "NIR_Red")
    b5_b4 = safe_ratio(red_edge1, red, "B5_B4")
    b6_b4 = safe_ratio(red_edge2, red, "B6_B4")
    b7_b4 = safe_ratio(red_edge3, red, "B7_B4")
    b8_b4 = safe_ratio(nir, red, "B8_B4")

    valid_mask = ee.Image(1).rename("valid_mask").updateMask(clear)
    # scl_water = scl.eq(6)
    scl_water = scl.eq(6).rename("scl_water").updateMask(clear)

    visible_brightness = (
        blue.add(green).add(red).divide(3)
        .rename("visible_brightness")
    )

    ndbi = (
        swir1.subtract(nir)
        .divide(swir1.add(nir))
        .rename("NDBI")
    )
    aweish = (
      blue
      .add(green.multiply(2.5))
      .subtract(nir.add(swir1).multiply(1.5))
      .subtract(swir2.multiply(0.25))
      .rename("AWEIsh")
    )

    # Formula-derived proxy features adapted from scripts.js.
    # They are kept separate from measured ground observations by the *_formula_s2_script names.
    b1_b8 = safe_ratio(coastal, nir, "tmp_b1_b8")
    ph_formula_s2_script = (
        ee.Image.constant(11.6424)
        .add(coastal.multiply(-216.0938))       # B1
        .add(green.multiply(28.5913))           # B3
        .add(red_edge2.multiply(368.9336))      # B6
        .add(nir.multiply(-239.9383))           # B8
        .rename("ph_formula_s2_script")
    )
    # ph_formula_s2_script = ph_formula_s2_script.updateMask(ph_formula_s2_script.lt(100))

    turbidity_formula_s2_script = (
        ee.Image.constant(0.4532)
        .add(coastal.multiply(-56.9454))         # B1
        .add(green.multiply(43.5723))            # B3
        .rename("turbidity_formula_s2_script")
    )
    # turbidity_formula_s2_script = turbidity_formula_s2_script.updateMask(
    #     turbidity_formula_s2_script.lt(100)
    # )

    salinity_formula_s2_script = safe_ratio(
        swir1.subtract(swir2),
        swir1.add(swir2),
        "salinity_formula_s2_script",
    )
    # salinity_formula_s2_script = salinity_formula_s2_script.updateMask(
    #     salinity_formula_s2_script.lt(100)
    # )

    b4_b6_interaction = (
        red.multiply(192.3651)
        .multiply(red_edge2.multiply(36.4049))
    )

    do_formula_s2_script = (
        ee.Image.constant(8.9055)
        .add(blue.multiply(-129.5866))
        .add(red.multiply(192.3651))
        .add(red_edge2.multiply(36.4049))
        .add(swir1.multiply(-116.7094))
        .rename("do_formula_s2_script")
    )

    chla_formula_s2_ndci_script = (
        ee.Image.constant(26.447)
        .add(blue.multiply(-1672.777))           # B2
        .add(green.multiply(266.620))            # B3
        .add(red.multiply(1402.560))             # B4
        .add(red_edge1.multiply(-58.610))        # B5
        .rename("chla_formula_s2_ndci_script")
    )
    # chla_formula_s2_ndci_script = chla_formula_s2_ndci_script.updateMask(
    #     chla_formula_s2_ndci_script.lt(100)
    # )

    blue_red_ratio = safe_ratio(blue, red, "tmp_blue_red_ratio").updateMask(
        blue.gt(0).And(red.gt(0))
    )
    ln_mosd = ee.Image(1.4856).multiply(blue_red_ratio.log()).add(ee.Image(0.2734))
    mosd = ee.Image(10).pow(ln_mosd)
    secchi_formula_s2_script = (
        ee.Image.constant(391.9)
        .add(coastal.multiply(22403.1))          # B1
        .add(green.multiply(-16175.8))           # B3
        .rename("secchi_formula_s2_script")
    )

    # secchi_formula_s2_script = secchi_formula_s2_script.updateMask(
    #     secchi_formula_s2_script.lt(10)
    # )

    chla_positive = chla_formula_s2_ndci_script.updateMask(
        chla_formula_s2_ndci_script.gt(0)
    )
    tsi_formula_s2_script = (
        ee.Image(30.6)
        .add(ee.Image(9.81).multiply(chla_positive.log()))
        .rename("tsi_formula_s2_script")
    )
    # tsi_formula_s2_script = tsi_formula_s2_script.updateMask(tsi_formula_s2_script.lt(200))

    built_or_bridge = (
        # concrete / road / roof: สว่าง และไม่ใช่น้ำชัด
        visible_brightness.gt(0.12)
        .And(mndwi.lt(0.12))
        .And(ndwi.lt(0.12))
        .And(aweish.lt(0.08))
    ).Or(
        # built-up / hard surface จาก SWIR
        ndbi.gt(0.00)
        .And(mndwi.lt(0.15))
    ).Or(
        # asphalt / bridge shadow บางแบบ: dark แต่ NIR/SWIR ไม่เหมือนน้ำดีพอ
        nir.gt(0.20)
        .And(swir1.gt(0.18))
        .And(mndwi.lt(-0.10))
        .And(scl.neq(6))
    )


    gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    jrc_water = gsw.select("occurrence").gte(70)



    jrc_current_water = (
        jrc_water
        .And(
            scl.eq(6)
            .Or(ndwi.gt(0.03))
            .Or(mndwi.gt(0.03))
            .Or(aweish.gt(0.03))
        )
        .And(nir.lt(0.25))
        .And(swir1.lt(0.18))
        .And(ndvi.lt(0.30))
        .And(built_or_bridge.Not())
    )

    strong_water = (
        mndwi.gt(0.20)
        .And(aweish.gt(0.05))
        .And(nir.lt(0.12))
        .And(swir1.lt(0.10))
        .And(ndvi.lt(0.15))
    )

    turbid_or_mixed_water = (
        mndwi.gt(-0.20) # ลดจาก 0.00 → -0.20
        .And(ndwi.gt(-0.30)) # ลดจาก -0.05 → -0.30
        .And(aweish.gt(-0.10))
        .And(nir.lt(0.20)) # เพิ่มจาก 0.16 → 0.20
        .And(swir1.lt(0.16)) # เพิ่มจาก 0.12 → 0.16
        .And(ndvi.lt(0.30)) # เพิ่มจาก 0.22 → 0.30
    )

    scl_current_water = (
        scl.eq(6)
        .And(nir.lt(0.16))
        .And(swir1.lt(0.12))
        .And(ndvi.lt(0.22))
        .And(
            mndwi.gt(0.00)
            .Or(ndwi.gt(-0.05))
            .Or(aweish.gt(-0.02))
        )
        .And(built_or_bridge.Not())
    )

    index_water_core = (
        strong_water
        .Or(turbid_or_mixed_water.And(mndwi.gt(0.05).Or(aweish.gt(0.02))))
    ).And(built_or_bridge.Not())

    water_core_raw = (
        scl_current_water
        .Or(jrc_current_water)
        .Or(index_water_core)
    )

    water_core = water_core_raw.And(built_or_bridge.Not())

    water_candidate = (
        turbid_or_mixed_water
        .And(built_or_bridge.Not())
    )

    core_neighborhood = water_core.focal_max(radius=2, units="pixels")

    water_mask = (
        water_core
        .Or(water_candidate.And(core_neighborhood))
        .And(built_or_bridge.Not())
        .rename("water_mask")
        .updateMask(clear)
    )


    out = (
        sr
        .addBands([
            ndwi, mndwi, ndti, ndci,
            red_green, red_blue, nir_red,
            b5_b4, b6_b4, b7_b4, b8_b4,
            ndvi, aweish, scl_water,
            ph_formula_s2_script,
            turbidity_formula_s2_script,
            salinity_formula_s2_script,
            do_formula_s2_script,
            chla_formula_s2_ndci_script,
            secchi_formula_s2_script,
            tsi_formula_s2_script,
        ])
        .addBands([valid_mask, water_mask])
        .updateMask(clear)
        .copyProperties(img, ["system:time_start", "system:index", "CLOUDY_PIXEL_PERCENTAGE"])
    )
    return out


def ee_dict_number(dictionary, key, default=0):
    value = ee.Dictionary(dictionary).get(key)
    return ee.Number(
        ee.Algorithms.If(
            ee.Algorithms.IsEqual(value, None),
            default,
            value,
        )
    )


def s2_collection_for_feature(feature, window_days, cloud_pct):
    field_date = ee.Date(feature.get("field_date"))
    geom = feature.geometry().buffer(BUFFER_M)
    start = field_date.advance(-window_days, "day")
    end = field_date.advance(window_days + 1, "day")

    def add_match_properties(img):
        signed_days = img.date().difference(field_date, "day")
        abs_days = signed_days.abs()
        cloud = ee.Number(img.get("CLOUDY_PIXEL_PERCENTAGE"))

        return img.set({
            "sat_date": img.date().format("YYYY-MM-dd"),
            "days_diff": abs_days,
            "days_diff_signed": signed_days,
            "image_id": img.get("system:index"),
            "scene_cloud_pct": cloud,
        })

    def add_local_buffer_qa(img):
        valid_count_dict = img.select("valid_mask").reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=geom,
            scale=SCALE_M,
            maxPixels=1e8,
            bestEffort=True,
            tileScale=4,
        )

        water_count_dict = img.select("water_mask").selfMask().reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=geom,
            scale=SCALE_M,
            maxPixels=1e8,
            bestEffort=True,
            tileScale=4,
        )

        valid_count = ee_dict_number(valid_count_dict, "valid_mask", 0)
        water_count = ee_dict_number(water_count_dict, "water_mask", 0)

        has_enough_water = water_count.gte(MIN_CANDIDATE_WATER_PIXELS)

        water_penalty = ee.Number(
            ee.Algorithms.If(has_enough_water, 0, 1000000)
        )

        no_clear_penalty = ee.Number(
            ee.Algorithms.If(valid_count.gt(0), 0, 2000000)
        )

        local_rank = (
            no_clear_penalty
            .add(water_penalty)
            .add(ee.Number(img.get("days_diff")).multiply(1000))
            .add(ee.Number(img.get("scene_cloud_pct")))
            .subtract(water_count.multiply(0.01))
        )

        return img.set({
            "candidate_valid_count": valid_count,
            "candidate_water_count": water_count,
            "candidate_has_enough_water": has_enough_water,
            "local_match_rank": local_rank,
        })

    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(geom)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct))
        .map(prep_s2)
        .map(add_match_properties)
        .map(add_local_buffer_qa)
        .sort("local_match_rank")
    )


def empty_s2_image():
    bands = S2_FEATURE_BANDS + ["valid_mask", "water_mask"]
    return ee.Image.constant([0] * len(bands)).rename(bands).updateMask(ee.Image(0))


## Session 9. Sentinel-2 Feature Extraction Function

Select the nearest Sentinel-2 image in the primary or fallback window and extract median band/index values plus QA fields.


In [ ]:
def dict_number(dictionary, key, default=0):
    value = ee.Dictionary(dictionary).get(key)
    return ee.Number(ee.Algorithms.If(value, value, default))


def extract_s2_for_feature(feature):
    buffer_geom = feature.geometry().buffer(BUFFER_M)

    coll_primary = s2_collection_for_feature(
        feature,
        window_days=PRIMARY_WINDOW_DAYS,
        cloud_pct=PRIMARY_CLOUD_PCT,
    )

    coll_fallback = s2_collection_for_feature(
        feature,
        window_days=FALLBACK_WINDOW_DAYS,
        cloud_pct=FALLBACK_CLOUD_PCT,
    )

    good_primary = coll_primary.filter(
        ee.Filter.gte("candidate_water_count", MIN_CANDIDATE_WATER_PIXELS)
    )

    good_fallback = coll_fallback.filter(
        ee.Filter.gte("candidate_water_count", MIN_CANDIDATE_WATER_PIXELS)
    )

    has_primary = coll_primary.size().gt(0)
    has_fallback = coll_fallback.size().gt(0)
    has_good_primary = good_primary.size().gt(0)
    has_good_fallback = good_fallback.size().gt(0)
    has_image = has_primary.Or(has_fallback)

    chosen_collection = ee.ImageCollection(
        ee.Algorithms.If(
            has_good_primary,
            good_primary,
            ee.Algorithms.If(
                has_good_fallback,
                good_fallback,
                ee.Algorithms.If(has_primary, coll_primary, coll_fallback),
            ),
        )
    )
    chosen_image = ee.Image(
        ee.Algorithms.If(has_image, chosen_collection.first(), empty_s2_image())
    )

    combined_reducer = ee.Reducer.median().combine(
        reducer2=ee.Reducer.count(),
        sharedInputs=True,
    )

    feature_img_clear = chosen_image.select(S2_FEATURE_BANDS)

    stats_clear = feature_img_clear.reduceRegion(
        reducer=combined_reducer,
        geometry=buffer_geom,
        scale=SCALE_M,
        maxPixels=1e8,
        tileScale=4,
    )

    valid_count_dict = chosen_image.select("valid_mask").selfMask().reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=buffer_geom,
        scale=SCALE_M,
        maxPixels=1e8,
        tileScale=4,
    )
    valid_count = dict_number(valid_count_dict, "valid_mask", 0)

    water_mask = chosen_image.select("water_mask").selfMask()
    feature_img_water = feature_img_clear.updateMask(water_mask)

    stats_water = feature_img_water.reduceRegion(
        reducer=combined_reducer,
        geometry=buffer_geom,
        scale=SCALE_M,
        maxPixels=1e8,
        tileScale=4,
    )

    median_pairs = ee.Dictionary.fromLists(
        S2_FEATURE_NAMES,
        ee.List(S2_FEATURE_BANDS).map(
            lambda band: stats_water.get(ee.String(band).cat("_median"))
        ),
    )

    water_count = dict_number(stats_water, "B3_count", 0)

    total_count_dict = (
        chosen_image.select("valid_mask")
        .unmask(1)
        .rename("total_pixel_count")
        .reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=buffer_geom,
            scale=SCALE_M,
            maxPixels=1e8,
            tileScale=4,
        )
    )

    total_count = dict_number(total_count_dict, "total_pixel_count", 0)

    water_fraction = ee.Algorithms.If(
        valid_count.gt(0),
        water_count.divide(valid_count).max(0).min(1),
        None,
    )

    local_cloud_fraction = ee.Algorithms.If(
        has_image.And(total_count.gt(0)),
        ee.Number(1).subtract(valid_count.divide(total_count)).max(0).min(1),
        None,
    )

    match_type = ee.Algorithms.If(
        has_good_primary,
        "window_3d",
        ee.Algorithms.If(
            has_good_fallback,
            "fallback_7d",
            ee.Algorithms.If(has_primary, "window_3d", ee.Algorithms.If(has_fallback, "fallback_7d", "no_match")),
        ),
    )

    selection_reason = ee.Algorithms.If(
        has_good_primary,
        "primary_clear_water",
        ee.Algorithms.If(
            has_good_fallback,
            "fallback_clear_water",
            ee.Algorithms.If(
                has_primary,
                "primary_no_enough_water",
                ee.Algorithms.If(has_fallback, "fallback_no_enough_water", "no_match"),
            ),
        ),
    )

    water_feature_reliable = water_count.gte(3)

    feature_extract_mode = ee.Algorithms.If(
        water_count.gt(0),
        "water_masked_pixels",
        "no_water_pixel",
    )

    meta = ee.Dictionary({
        "match_type": match_type,
        "sat_date": ee.Algorithms.If(has_image, chosen_image.get("sat_date"), None),
        "days_diff": ee.Algorithms.If(has_image, chosen_image.get("days_diff"), None),
        "days_diff_signed": ee.Algorithms.If(has_image, chosen_image.get("days_diff_signed"), None),
        "image_id": ee.Algorithms.If(has_image, chosen_image.get("image_id"), None),
        "scene_cloud_pct": ee.Algorithms.If(has_image, chosen_image.get("scene_cloud_pct"), None),
        "buffer_m": BUFFER_M,
        "scale_m": SCALE_M,
        "primary_window_days": PRIMARY_WINDOW_DAYS,
        "fallback_window_days": FALLBACK_WINDOW_DAYS,
        "valid_pixel_count": valid_count,
        "valid_water_pixel_count": water_count,
        "total_pixel_count": total_count,
        "water_pixel_fraction": water_fraction,
        "local_cloud_fraction": local_cloud_fraction,
        "water_feature_reliable": water_feature_reliable,
        "feature_extract_mode": feature_extract_mode,
        "candidate_valid_count": ee.Algorithms.If(has_image, chosen_image.get("candidate_valid_count"), None),
        "candidate_water_count": ee.Algorithms.If(has_image, chosen_image.get("candidate_water_count"), None),
        "candidate_has_enough_water": ee.Algorithms.If(has_image, chosen_image.get("candidate_has_enough_water"), None),
        "local_match_rank": ee.Algorithms.If(has_image, chosen_image.get("local_match_rank"), None),
        "selection_reason": selection_reason,
        "min_candidate_water_pixels": MIN_CANDIDATE_WATER_PIXELS,
    })

    return feature.set(median_pairs).set(meta)


s2_features_fc = ee_points.map(extract_s2_for_feature)
print("FeatureCollection ready:", s2_features_fc.size().getInfo())

"""
valid_pixel_count
= จำนวน pixel ใน buffer ที่ผ่าน clear mask

valid_water_pixel_count
= จำนวน pixel ใน buffer ที่ผ่าน clear mask และ NDWI > 0

total_pixel_count
= จำนวน pixel ทั้งหมดใน buffer ไม่สน clear mask
"""

## Session 10. Preview and Visual QA

Preview a few extracted rows and define a map function for any sample ID.


In [ ]:
def ee_fc_to_pandas_small(fc, limit=10):
    info = fc.limit(limit).getInfo()
    rows = [f["properties"] for f in info["features"]]
    return pd.DataFrame(rows)


test_features = ee_fc_to_pandas_small(s2_features_fc, limit=20)

preview_cols = [
    "sample_id",
    "station_canonical",
    "field_date",
    "sat_date",
    "days_diff",
    "match_type",
    "B3_median",
    "B4_median",
    "NDWI_median",
    "NDTI_median",
    "NDCI_median",
    "valid_pixel_count",
    "valid_water_pixel_count",
    "water_pixel_fraction",
    "local_cloud_fraction",
]

test_features[[c for c in preview_cols if c in test_features.columns]]

In [ ]:
def visualize_s2_match(sample_id):
    row = field[field["sample_id"] == sample_id].iloc[0]
    feature = row_to_ee_feature(row)

    coll_primary = s2_collection_for_feature(
        feature,
        window_days=PRIMARY_WINDOW_DAYS,
        cloud_pct=PRIMARY_CLOUD_PCT,
    )

    if coll_primary.size().getInfo() > 0:
        coll = coll_primary
        match_type = "window_3d"
    else:
        coll = s2_collection_for_feature(
            feature,
            window_days=FALLBACK_WINDOW_DAYS,
            cloud_pct=FALLBACK_CLOUD_PCT,
        )
        match_type = "fallback_7d"

    if coll.size().getInfo() == 0:
        print("No Sentinel-2 image match for:", sample_id)
        return None

    img = ee.Image(coll.first())
    buffer_geom = feature.geometry().buffer(BUFFER_M)

    center = [row["latitude"], row["longitude"]]
    m = geemap.Map(center=center, zoom=15)

    rgb_vis = {
        "bands": ["B4", "B3", "B2"],
        "min": 0,
        "max": 0.3,
        "gamma": 1.2,
    }

    ndwi_vis = {
        "min": -0.5,
        "max": 0.8,
        "palette": ["brown", "white", "blue"],
    }

    index_vis = {
        "min": -0.3,
        "max": 0.3,
        "palette": ["purple", "white", "green"],
    }

    mask_vis = {
        "min": 0,
        "max": 1,
        "palette": ["white", "cyan"],
    }

    water_only = (
    img.select("water_mask")
    .selfMask()
    )

    clear_non_water = (
        img.select("valid_mask").unmask(0).eq(1)
        .And(img.select("water_mask").unmask(0).eq(0))
        .selfMask()
    )

    m.addLayer(img, rgb_vis, "Sentinel-2 RGB")
    # m.addLayer(img.select("NDWI"), ndwi_vis, "NDWI")
    # m.addLayer(img.select("MNDWI"), ndwi_vis, "MNDWI")
    # m.addLayer(img.select("NDTI"), index_vis, "NDTI")
    # m.addLayer(img.select("NDCI"), index_vis, "NDCI")
    # m.addLayer(img.select("water_mask"), mask_vis, f"Water mask ")
    m.addLayer(water_only,{"palette": ["cyan"]},"Water mask only")
    m.addLayer(img.select("valid_mask").selfMask(),{"palette": ["yellow"]},"Valid clear pixels")
    m.addLayer(clear_non_water,{"palette": ["lightgray"]},"Clear non-water")
    m.addLayer(buffer_geom, {"color": "yellow"}, f"{BUFFER_M} m buffer")


    popup = f"""
    sample_id: {sample_id}<br>
    station: {row["station_canonical"]}<br>
    field_date: {row["survey_date"]}<br>
    match_type: {match_type}<br>
    sat_date: {img.get("sat_date").getInfo()}<br>
    days_diff: {img.get("days_diff").getInfo()}<br>
    scene_cloud_pct: {img.get("scene_cloud_pct").getInfo()}
    """

    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=folium.Popup(popup, max_width=350),
        tooltip=sample_id,
    ).add_to(m)

    return m

## Session 11. Export Schema

Fill missing Earth Engine properties so every chunk CSV has consistent columns.


In [ ]:
S2_EXPORT_COLS = [
    "sample_id",
    "station_survey_id",
    "canonical_station_survey_id",
    "station_original",
    "station_canonical",
    "survey_no",
    "year_be",
    "year_ad",
    "field_date",
    "lat",
    "lon",
    "water_body",
    "coordinate_waterbody",
    "coordinate_assignment_status",
    "sat_date",
    "days_diff",
    "days_diff_signed",
    "match_type",
    "image_id",
    "scene_cloud_pct",
    "buffer_m",
    "scale_m",
    "primary_window_days",
    "fallback_window_days",
    "valid_pixel_count",
    "valid_water_pixel_count",
    "total_pixel_count",
    "water_pixel_fraction",
    "local_cloud_fraction",
    "selection_reason",
    "candidate_valid_count",
    "candidate_water_count",
    "candidate_has_enough_water",
    "local_match_rank",
    "min_candidate_water_pixels",
] + S2_FEATURE_NAMES

# geemap validates selectors against the FeatureCollection schema.
# The first rows may be no_match, so band/date properties can be missing unless we add defaults first.
S2_STRING_COLS = [
    "sample_id",
    "station_survey_id",
    "canonical_station_survey_id",
    "station_original",
    "station_canonical",
    "survey_no",
    "field_date",
    "water_body",
    "coordinate_waterbody",
    "coordinate_assignment_status",
    "sat_date",
    "match_type",
    "image_id",
    "selection_reason",
]

S2_NUMERIC_COLS = [col for col in S2_EXPORT_COLS if col not in S2_STRING_COLS]
S2_NODATA = -9999

def _ee_values_with_defaults(feature, cols, default_value):
    col_list = ee.List(cols)

    def value_or_default(col):
        col = ee.String(col)
        value = feature.get(col)
        return ee.Algorithms.If(
            ee.Algorithms.IsEqual(value, None),
            default_value,
            value,
        )

    return ee.Dictionary.fromLists(col_list, col_list.map(value_or_default))


def ensure_export_schema(feature):
    string_values = _ee_values_with_defaults(feature, S2_STRING_COLS, "")
    numeric_values = _ee_values_with_defaults(feature, S2_NUMERIC_COLS, S2_NODATA)
    return feature.set(string_values).set(numeric_values)


# D. Export GGE to Drive

## Session 12. Chunked Earth Engine Export

Use chunked Drive export to avoid Earth Engine memory limits. Keep the export switch off unless intentionally starting new tasks.


In [ ]:
field_chunks = field.reset_index(drop=True).copy()
field_chunks["chunk_id"] = (field_chunks.index // CHUNK_SIZE).astype(int)

chunk_plan = field_chunks.groupby("chunk_id").size().rename("rows").reset_index()
print("Total rows:", len(field_chunks))
print("Chunks:", len(chunk_plan))
display(chunk_plan)


In [ ]:
# Safety switch: keep False unless you intentionally want to start new Earth Engine export tasks.
START_EE_EXPORT = False

chunk_tasks = []

if START_EE_EXPORT:
    for chunk_id, chunk_df in field_chunks.groupby("chunk_id"):
        chunk_id_int = int(chunk_id)

        ee_chunk_points = pandas_to_ee_points(chunk_df.drop(columns=["chunk_id"]))
        ee_chunk_features = (
            ee_chunk_points
            .map(extract_s2_for_feature)
            .map(ensure_export_schema)
            .select(S2_EXPORT_COLS)
        )

        task = ee.batch.Export.table.toDrive(
            collection=ee_chunk_features,
            description=f"s2_{RUN_TAG}_chunk_{chunk_id_int:02d}",
            folder=DRIVE_CHUNK_FOLDER,
            fileNamePrefix=f"{DRIVE_CHUNK_PREFIX}_{chunk_id_int:02d}",
            fileFormat="CSV",
            selectors=S2_EXPORT_COLS,
        )
        task.start()

        chunk_tasks.append({
            "chunk_id": chunk_id_int,
            "rows": len(chunk_df),
            "task_id": task.id,
            "task": task,
        })

        print(f"Started chunk {chunk_id_int:02d}: rows={len(chunk_df)}, task_id={task.id}")
else:
    print("START_EE_EXPORT is False. Set it to True only when you want to start new export tasks.")


## Session 13. Monitor or Recover Export Tasks

Check task status during the same runtime or recover task status after Colab disconnects.


In [ ]:
def chunk_task_status(chunk_tasks):
    rows = []
    for item in chunk_tasks:
        status = item["task"].status()
        rows.append({
            "chunk_id": item["chunk_id"],
            "rows": item["rows"],
            "task_id": item["task_id"],
            "state": status.get("state"),
            "error_message": status.get("error_message", ""),
        })
    return pd.DataFrame(rows).sort_values("chunk_id")


def list_s2_chunk_tasks(prefix=None):
    if prefix is None:
        prefix = f"s2_{RUN_TAG}_chunk_"

    rows = []
    for task in ee.batch.Task.list():
        status = task.status()
        description = status.get("description", "")
        if description.startswith(prefix):
            rows.append({
                "description": description,
                "task_id": status.get("id"),
                "state": status.get("state"),
                "creation_timestamp_ms": status.get("creation_timestamp_ms"),
                "update_timestamp_ms": status.get("update_timestamp_ms"),
                "error_message": status.get("error_message", ""),
            })
    out = pd.DataFrame(rows)
    if len(out):
        out = out.sort_values(["description", "creation_timestamp_ms"]).reset_index(drop=True)
    return out

if chunk_tasks:
    display(chunk_task_status(chunk_tasks))
else:
    display(list_s2_chunk_tasks())


## Session 14. Retry One Missing Chunk

Retry only a missing/failed chunk instead of re-exporting the whole dataset.


In [ ]:
RETRY_CHUNK_ID = None  # Example: 1

if RETRY_CHUNK_ID is not None:
    retry_df = field_chunks[field_chunks["chunk_id"].eq(RETRY_CHUNK_ID)].drop(columns=["chunk_id"])
    print("Retry chunk:", RETRY_CHUNK_ID, "rows:", len(retry_df))

    retry_points = pandas_to_ee_points(retry_df)
    retry_features = (
        retry_points
        .map(extract_s2_for_feature)
        .map(ensure_export_schema)
        .select(S2_EXPORT_COLS)
    )

    retry_task = ee.batch.Export.table.toDrive(
        collection=retry_features,
        description=f"s2_{RUN_TAG}_chunk_{RETRY_CHUNK_ID:02d}_retry",
        folder=DRIVE_CHUNK_FOLDER,
        fileNamePrefix=f"{DRIVE_CHUNK_PREFIX}_{RETRY_CHUNK_ID:02d}_retry",
        fileFormat="CSV",
        selectors=S2_EXPORT_COLS,
    )
    retry_task.start()
    print("Started retry task:", retry_task.id)
else:
    print("No retry requested. Set RETRY_CHUNK_ID to an integer chunk id if needed.")


## Session 15. Merge Chunk CSV Files

Mount Drive, read all chunk CSV files, restore row order, and save a raw merged Sentinel-2 feature CSV.


In [ ]:
drive.mount("/content/drive", force_remount=False)

chunk_dir = Path("/content/drive/MyDrive") / DRIVE_CHUNK_FOLDER
chunk_dir.mkdir(parents=True, exist_ok=True)

raw_chunk_files = sorted(chunk_dir.glob(f"{DRIVE_CHUNK_PREFIX}_*.csv"))
print("chunk_dir:", chunk_dir)
print("raw chunk files found:", len(raw_chunk_files))
for file_path in raw_chunk_files:
    print(file_path.name, file_path.stat().st_size)

chunk_file_pattern = re.compile(rf"^{re.escape(DRIVE_CHUNK_PREFIX)}_(\d+).*\.csv$")
chunk_file_rows = []
for file_path in raw_chunk_files:
    match = chunk_file_pattern.match(file_path.name)
    if match:
        chunk_file_rows.append({
            "chunk_id": int(match.group(1)),
            "path": file_path,
            "name": file_path.name,
            "size_bytes": file_path.stat().st_size,
            "modified_time": file_path.stat().st_mtime,
        })

chunk_files_df = pd.DataFrame(chunk_file_rows)
print("parsed chunk files:", len(chunk_files_df))
if len(chunk_files_df):
    display(chunk_files_df.sort_values(["chunk_id", "modified_time"]))

expected_chunks = set(chunk_plan["chunk_id"].astype(int))
found_chunks = set(chunk_files_df["chunk_id"].astype(int)) if len(chunk_files_df) else set()
missing_chunks = sorted(expected_chunks - found_chunks)

if missing_chunks:
    print("Missing chunk CSV files:", missing_chunks)
    display(list_s2_chunk_tasks())
    raise RuntimeError("Some chunk CSV files are missing. Wait for tasks to complete or retry missing chunks.")

latest_chunk_files = (
    chunk_files_df
    .sort_values("modified_time")
    .groupby("chunk_id", as_index=False)
    .tail(1)
    .sort_values("chunk_id")
)

chunk_dfs = []
for _, row in latest_chunk_files.iterrows():
    temp = pd.read_csv(row["path"])
    temp["source_chunk_file"] = row["name"]
    chunk_dfs.append(temp)

s2_features = pd.concat(chunk_dfs, ignore_index=True)

for col in S2_NUMERIC_COLS:
    if col in s2_features.columns:
        s2_features[col] = pd.to_numeric(s2_features[col], errors="coerce").replace(S2_NODATA, np.nan)

for col in S2_STRING_COLS:
    if col in s2_features.columns:
        s2_features[col] = s2_features[col].replace("", np.nan)

field_order = field[["sample_id"]].copy()
field_order["_field_order"] = np.arange(len(field_order))

s2_features = (
    s2_features
    .drop_duplicates("sample_id", keep="last")
    .merge(field_order, on="sample_id", how="left")
    .sort_values("_field_order")
    .drop(columns=["_field_order"])
    .reset_index(drop=True)
)

s2_raw_out = OUT_DIR / f"sentinel2_features_raw_{RUN_TAG}.csv"
s2_features.to_csv(s2_raw_out, index=False, encoding="utf-8-sig")

print("merged s2_features shape:", s2_features.shape)
print("expected rows:", len(field))
print("row count ok:", len(s2_features) == len(field))
print("saved raw merged CSV:", s2_raw_out)
s2_features.head()


## Session 16. Add Match and Quality Flags

Create exact-date, match-window, cloud, water-pixel, and usability flags.


In [ ]:
def add_s2_quality_columns(df):
    out = df.copy()
    out["field_date_dt"] = pd.to_datetime(out["field_date"], errors="coerce")
    out["sat_date_dt"] = pd.to_datetime(out["sat_date"], errors="coerce")

    out["abs_date_diff_days"] = (
        out["sat_date_dt"].dt.normalize()
        .sub(out["field_date_dt"].dt.normalize())
        .abs()
        .dt.days
    )

    out["exact_date_match"] = (
        out["sat_date_dt"].notna()
        & out["field_date_dt"].dt.date.eq(out["sat_date_dt"].dt.date)
    )

    def classify_match_window(row):
        if pd.isna(row["sat_date_dt"]) or row["match_type"] == "no_match":
            return "no_match"
        if row["abs_date_diff_days"] == 0:
            return "exact_date"
        if row["abs_date_diff_days"] <= 1:
            return "within_1_day"
        if row["abs_date_diff_days"] <= 3:
            return "within_3_days"
        return "fallback_7d"

    out["match_window"] = out.apply(classify_match_window, axis=1)
    out["local_cloud_pct"] = out["local_cloud_fraction"] * 100
    out["water_pixel_pct"] = out["water_pixel_fraction"] * 100

    out["has_image_match"] = out["match_type"].ne("no_match")
    out["has_band_value"] = out["B3_median"].notna()
    out["has_clear_pixel"] = out["valid_pixel_count"].fillna(0).gt(0)
    out["has_water_pixel"] = out["valid_water_pixel_count"].fillna(0).gt(0)
    out["usable_s2"] = out["has_image_match"] & out["has_band_value"] & out["has_clear_pixel"]
    out["usable_s2_water"] = out["usable_s2"] & out["has_water_pixel"]

    out["data_quality"] = np.select(
        [
            ~out["has_image_match"],
            out["has_image_match"] & ~out["has_clear_pixel"],
            out["has_clear_pixel"] & ~out["has_water_pixel"],
            out["has_clear_pixel"] & out["has_water_pixel"],
        ],
        [
            "no_match",
            "matched_no_clear_pixel",
            "clear_but_no_water_pixel",
            "has_water_pixel",
        ],
        default="unknown",
    )
    return out


s2_features = add_s2_quality_columns(s2_features)
s2_out = OUT_DIR / f"sentinel2_features_{RUN_TAG}.csv"
s2_features.to_csv(s2_out, index=False, encoding="utf-8-sig")

print("Saved QA-updated Sentinel-2 CSV:", s2_out)
print("Rows:", len(s2_features))
print("Exact date matches:", int(s2_features["exact_date_match"].sum()))
print("Usable S2 rows:", int(s2_features["usable_s2"].sum()))
display(s2_features.head())


## Session 17. QA Summary Tables

Summarize match timing and data quality by BE year.


In [ ]:
print("Match type counts:")
print(s2_features["match_type"].value_counts(dropna=False))

print("\nMatch window by year:")
display(s2_features.groupby(["year_be", "match_window"]).size().unstack(fill_value=0))

print("\nData quality by year:")
display(s2_features.groupby(["year_be", "data_quality"]).size().unstack(fill_value=0))

usable_summary = (
    s2_features
    .groupby("year_be")
    .agg(
        total=("sample_id", "count"),
        matched=("has_image_match", "sum"),
        exact_date=("exact_date_match", "sum"),
        has_clear_pixel=("has_clear_pixel", "sum"),
        has_water_pixel=("has_water_pixel", "sum"),
        usable_s2=("usable_s2", "sum"),
        median_local_cloud_pct=("local_cloud_pct", "median"),
        median_water_pixel_pct=("water_pixel_pct", "median"),
    )
    .reset_index()
)

usable_summary["matched_pct"] = usable_summary["matched"] / usable_summary["total"] * 100
usable_summary["usable_pct"] = usable_summary["usable_s2"] / usable_summary["total"] * 100
usable_summary["water_pixel_row_pct"] = usable_summary["has_water_pixel"] / usable_summary["total"] * 100
display(usable_summary)


## Session 18. QA Plots

Plot match timing, data quality, and local cloud/masked-pixel metrics.


In [ ]:
match_order = ["no_match", "exact_date", "within_1_day", "within_3_days", "fallback_7d"]
match_counts = (
    s2_features
    .groupby(["year_be", "match_window"])
    .size()
    .reset_index(name="count")
)
match_counts["match_window"] = pd.Categorical(match_counts["match_window"], categories=match_order, ordered=True)

plt.figure(figsize=(11, 5))
sns.barplot(data=match_counts, x="year_be", y="count", hue="match_window", hue_order=match_order)
plt.title("Sentinel-2 Match Timing by Year")
plt.xlabel("Year BE")
plt.ylabel("Number of samples")
plt.legend(title="Match timing", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
quality_counts = (
    s2_features
    .groupby(["year_be", "data_quality"])
    .size()
    .reset_index(name="count")
)

quality_pivot = quality_counts.pivot(index="year_be", columns="data_quality", values="count").fillna(0)

ax = quality_pivot.plot(kind="bar", stacked=True, figsize=(11, 5))
ax.set_title("Sentinel-2 Data Quality by Year")
ax.set_xlabel("Year BE")
ax.set_ylabel("Number of samples")
ax.legend(title="Data quality", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=s2_features[s2_features["has_image_match"]], x="year_be", y="local_cloud_pct", ax=axes[0])
axes[0].set_title("Local Cloud/Masked Pixel Percentage by Year")
axes[0].set_xlabel("Year BE")
axes[0].set_ylabel("Local cloud/masked pixels (%)")
axes[0].set_ylim(0, 100)

sns.scatterplot(
    data=s2_features[s2_features["has_image_match"]],
    x="scene_cloud_pct",
    y="local_cloud_pct",
    hue="data_quality",
    alpha=0.7,
    ax=axes[1],
)
axes[1].set_title("Scene Cloud vs Local Cloud/Masked Pixels")
axes[1].set_xlabel("Scene cloud (%)")
axes[1].set_ylabel("Local cloud/masked pixels (%)")
axes[1].set_xlim(0, 100)
axes[1].set_ylim(0, 100)
axes[1].legend(title="Data quality", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Session 19. Build Model-Ready Sentinel-2 Dataset

Join field target values with Sentinel-2 features and QA fields.


In [ ]:
field_label_table = field[["sample_id"] + field_value_cols].copy()
model_dataset = field_label_table.merge(s2_features, on="sample_id", how="right")

model_dataset_out = OUT_DIR / f"sentinel2_model_dataset_{RUN_TAG}.csv"
model_dataset.to_csv(model_dataset_out, index=False, encoding="utf-8-sig")

script_formula_feature_cols = [
    f"{band}_median" for band in S2_SCRIPT_FORMULA_BANDS
    if f"{band}_median" in model_dataset.columns
]
formula_dataset_cols = [
    col for col in metadata_cols + field_value_cols
    if col in model_dataset.columns
] + [
    col for col in [
        "field_date", "sat_date", "days_diff", "days_diff_signed",
        "match_type", "match_window", "data_quality",
        "valid_water_pixel_count", "water_pixel_fraction", "local_cloud_fraction",
    ]
    if col in model_dataset.columns
] + script_formula_feature_cols

formula_dataset = model_dataset.loc[:, pd.Index(formula_dataset_cols).drop_duplicates()].copy()
formula_dataset_out = OUT_DIR / f"sentinel2_script_formula_features_{RUN_TAG}.csv"
formula_dataset.to_csv(formula_dataset_out, index=False, encoding="utf-8-sig")

print("Model dataset shape:", model_dataset.shape)
print("Saved:", model_dataset_out)
print("Script formula dataset shape:", formula_dataset.shape)
print("Saved formula features:", formula_dataset_out)
print("Target availability in all model rows:")
display(model_dataset[target_cols].notna().sum().rename("non_missing_targets"))


## Session 19B. Compare Script-Formula Features with Ground Observations

Create scatter plots and metrics only for formula-derived parameters that have a matching ground-observation column in the field table.


In [ ]:
from sklearn.metrics import r2_score
FORMULA_COMPARISON_SPECS = [
    {
        "parameter": "pH",
        "truth_candidates": ["pH_clean"],
        "formula_col": "ph_formula_s2_script_median",
        "unit": "pH",
    },
    {
        "parameter": "Turbidity",
        "truth_candidates": ["turbidity_NTU_clean"],
        "formula_col": "turbidity_formula_s2_script_median",
        "unit": "NTU",
    },
    {
        "parameter": "DO",
        "truth_candidates": ["DO_mg_l_clean"],
        "formula_col": "do_formula_s2_script_median",
        "unit": "mg/L",
    },
    {
        "parameter": "Salinity",
        "truth_candidates": ["salinity_ppt_clean"],
        "formula_col": "salinity_formula_s2_script_median",
        "unit": "ppt",
    },
    {
        "parameter": "Chlorophyll-a",
        "truth_candidates": [
            "chlorophyll_a_clean",
            "chlorophyll_a_ug_l_clean",
            "chlorophyll_a_mg_m3_clean",
            "chla_clean",
            "chla_ug_l_clean",
            "Chl_a_clean",
            "Chl-a_clean",
        ],
        "formula_col": "chla_formula_s2_ndci_script_median",
        "unit": "ug/L or mg/m3",
    },
    {
        "parameter": "Secchi depth",
        "truth_candidates": [
            "secchi_depth_m_clean",
            "secchi_m_clean",
            "SDD_m_clean",
            "SDD_clean",
            "transparency_m_clean",
        ],
        "formula_col": "secchi_formula_s2_script_median",
        "unit": "m",
    },
    {
        "parameter": "TSI",
        "truth_candidates": ["TSI_clean", "trophic_state_index_clean"],
        "formula_col": "tsi_formula_s2_script_median",
        "unit": "index",
    },
]


def first_existing_col(candidates, columns):
    for col in candidates:
        if col in columns:
            return col
    return None


formula_comparison_frames = []
formula_comparison_metrics = []

for spec in FORMULA_COMPARISON_SPECS:
    truth_col = first_existing_col(spec["truth_candidates"], model_dataset.columns)
    formula_col = spec["formula_col"]

    if truth_col is None or formula_col not in model_dataset.columns:
        continue

    base_cols = [
        col for col in [
            "sample_id", "station_canonical", "year_be", "field_date", "sat_date",
            "match_type", "match_window", "data_quality",
            "valid_water_pixel_count", "water_pixel_fraction", "local_cloud_fraction",
            truth_col, formula_col,
        ]
        if col in model_dataset.columns
    ]
    sub = model_dataset[base_cols].copy()
    sub[truth_col] = pd.to_numeric(sub[truth_col], errors="coerce")
    sub[formula_col] = pd.to_numeric(sub[formula_col], errors="coerce")
    sub = sub.dropna(subset=[truth_col, formula_col])
    sub = sub[np.isfinite(sub[truth_col]) & np.isfinite(sub[formula_col])]

    if sub.empty:
        continue

    truth = sub[truth_col].to_numpy(dtype=float)
    pred = sub[formula_col].to_numpy(dtype=float)

    diff = pred - truth
    corr = sub[[truth_col, formula_col]].corr().iloc[0, 1] if len(sub) >= 2 else np.nan
    predictive_r2 = r2_score(truth, pred) if len(sub) >= 2 else np.nan

    formula_comparison_metrics.append({
        "parameter": spec["parameter"],
        "ground_truth_col": truth_col,
        "formula_col": formula_col,
        "unit": spec["unit"],
        "n_pairs": len(sub),
        "pearson_r": corr,
        "corr_squared": corr ** 2 if pd.notna(corr) else np.nan,
        "predictive_r2": predictive_r2,
        "bias_formula_minus_ground": diff.mean(),
        "mae": np.mean(np.abs(diff)),
        "rmse": np.sqrt(np.mean(diff ** 2)),
        "ground_median": sub[truth_col].median(),
        "formula_median": sub[formula_col].median(),
    })

    formula_comparison_frames.append(
        sub.rename(columns={truth_col: "ground_truth", formula_col: "formula_value"})
        .assign(
            parameter=spec["parameter"],
            ground_truth_col=truth_col,
            formula_col=formula_col,
            unit=spec["unit"],
        )
    )

formula_comparison_long_out = OUT_DIR / f"sentinel2_script_formula_vs_ground_truth_long_{RUN_TAG}.csv"
formula_comparison_metrics_out = OUT_DIR / f"sentinel2_script_formula_vs_ground_truth_metrics_{RUN_TAG}.csv"
formula_comparison_plot_out = OUT_DIR / f"sentinel2_script_formula_vs_ground_truth_{RUN_TAG}.png"

if formula_comparison_frames:
    formula_comparison_long = pd.concat(formula_comparison_frames, ignore_index=True)
    formula_comparison_metrics_df = pd.DataFrame(formula_comparison_metrics)

    formula_comparison_long.to_csv(formula_comparison_long_out, index=False, encoding="utf-8-sig")
    formula_comparison_metrics_df.to_csv(formula_comparison_metrics_out, index=False, encoding="utf-8-sig")

    display(formula_comparison_metrics_df)

    plot_params = formula_comparison_metrics_df["parameter"].tolist()
    n_plots = len(plot_params)
    n_cols = min(3, n_plots)
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows), squeeze=False)
    axes = axes.ravel()

    for ax, parameter in zip(axes, plot_params):
        plot_df = formula_comparison_long[formula_comparison_long["parameter"].eq(parameter)].copy()
        metric_row = formula_comparison_metrics_df[
            formula_comparison_metrics_df["parameter"].eq(parameter)
        ].iloc[0]

        hue_col = "match_window" if "match_window" in plot_df.columns else None
        sns.scatterplot(
            data=plot_df,
            x="ground_truth",
            y="formula_value",
            hue=hue_col,
            alpha=0.75,
            s=45,
            ax=ax,
        )

        xy = pd.concat([plot_df["ground_truth"], plot_df["formula_value"]]).dropna()
        if len(xy):
            low = xy.min()
            high = xy.max()
            if np.isfinite(low) and np.isfinite(high) and low != high:
                pad = (high - low) * 0.05
                ax.plot([low - pad, high + pad], [low - pad, high + pad], "--", color="gray", linewidth=1)
                ax.set_xlim(low - pad, high + pad)
                ax.set_ylim(low - pad, high + pad)

        ax.set_title(
            f"{parameter}: ground truth vs script formula\n"
            f"n={int(metric_row['n_pairs'])}, "
            f"r={metric_row['pearson_r']:.2f}, "
            f"R2={metric_row['predictive_r2']:.2f}, "
            f"RMSE={metric_row['rmse']:.2f}"
        )
        ax.set_xlabel(f"Ground truth ({metric_row['ground_truth_col']})")
        ax.set_ylabel(f"Formula value ({metric_row['formula_col']})")
        if hue_col:
            ax.legend(title=hue_col, fontsize=8)

    for ax in axes[n_plots:]:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(formula_comparison_plot_out, dpi=180, bbox_inches="tight")
    plt.show()

    print("Saved formula comparison long table:", formula_comparison_long_out)
    print("Saved formula comparison metrics:", formula_comparison_metrics_out)
    print("Saved formula comparison plot:", formula_comparison_plot_out)
else:
    formula_comparison_long = pd.DataFrame()
    formula_comparison_metrics_df = pd.DataFrame()
    print("No formula/ground-truth pairs available for comparison plots.")


## Session 20. Strict and Relaxed Training Candidate Sets

Create candidate rows for model training. Strict rows are preferred for main experiments; relaxed rows are useful for sensitivity tests.


In [ ]:
strict_mask = (
    model_dataset["match_type"].eq("window_3d")
    & model_dataset["abs_date_diff_days"].le(3)
    & model_dataset["B3_median"].notna()
    & model_dataset["valid_pixel_count"].fillna(0).gt(0)
    & model_dataset["valid_water_pixel_count"].fillna(0).gt(0)
    & model_dataset["water_pixel_fraction"].fillna(0).ge(0.10)
    & model_dataset["local_cloud_fraction"].fillna(1).le(0.50)
)

relaxed_mask = (
    model_dataset["match_type"].isin(["window_3d", "fallback_7d"])
    & model_dataset["abs_date_diff_days"].le(7)
    & model_dataset["B3_median"].notna()
    & model_dataset["valid_pixel_count"].fillna(0).gt(0)
    & model_dataset["valid_water_pixel_count"].fillna(0).gt(0)
    & model_dataset["water_pixel_fraction"].fillna(0).gt(0)
    & model_dataset["local_cloud_fraction"].fillna(1).le(0.70)
)

train_strict = model_dataset[strict_mask].copy()
train_relaxed = model_dataset[relaxed_mask].copy()

train_strict_out = OUT_DIR / f"sentinel2_train_strict_candidates_{RUN_TAG}.csv"
train_relaxed_out = OUT_DIR / f"sentinel2_train_relaxed_candidates_{RUN_TAG}.csv"
train_strict.to_csv(train_strict_out, index=False, encoding="utf-8-sig")
train_relaxed.to_csv(train_relaxed_out, index=False, encoding="utf-8-sig")

print("Strict train candidates:", train_strict.shape)
print("Relaxed train candidates:", train_relaxed.shape)
print("Saved strict:", train_strict_out)
print("Saved relaxed:", train_relaxed_out)

print("\nStrict target availability:")
display(train_strict[target_cols].notna().sum().rename("non_missing_targets"))
print("\nRelaxed target availability:")
display(train_relaxed[target_cols].notna().sum().rename("non_missing_targets"))


## Session 21. Copy Outputs to Final Google Drive Folder

Copy merged outputs and chunks to the desired nested Drive folder after export/merge succeeds.


In [ ]:
points_out = OUT_DIR / "rs_match_points.csv"
points.to_csv(points_out, index=False, encoding="utf-8-sig")
print("Saved:", points_out)

In [ ]:
if COPY_OUTPUTS_TO_FINAL_DRIVE:
    FINAL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

    output_files = [
        points_out,
        s2_raw_out,
        s2_out,
        model_dataset_out,
        train_strict_out,
        train_relaxed_out,
    ]

    for optional_output_name in [
        "formula_dataset_out",
        "formula_comparison_long_out",
        "formula_comparison_metrics_out",
        "formula_comparison_plot_out",
    ]:
        if optional_output_name in globals():
            output_files.append(globals()[optional_output_name])

    for src in output_files:
        if Path(src).exists():
            dst = FINAL_DRIVE_DIR / Path(src).name
            shutil.copy2(src, dst)
            print("copied:", dst)

    copied_chunks = 0
    for src in sorted(chunk_dir.glob(f"{DRIVE_CHUNK_PREFIX}_*.csv")):
        dst = FINAL_DRIVE_DIR / src.name
        shutil.copy2(src, dst)
        copied_chunks += 1

    print("copied chunk files:", copied_chunks)
    print("Final Drive folder:", FINAL_DRIVE_DIR)
else:
    print("COPY_OUTPUTS_TO_FINAL_DRIVE is False.")
